# Energy Indicators — Run Guide

**Collection ID:** `EO.BSC.DAT.ENERGY_INDICATORS`
**Owner:** Francesc Roura Adserias (BSC)

Energy-relevant indicators from the Climate Adaptation Digital Twin — wind energy
(capacity factors, wind histograms, low/high wind events), energy demand
(heating/cooling degree-days), and a solar indicator (photovoltaic potential).
Global, ~5-10km resolution, derived from hourly IFS-NEMO output under SSP3-7.0.

This notebook runs the **whole pipeline against real Energy Indicators example
data** into a throwaway sandbox directory, and stops **before** the upload step.
No credentials needed, nothing is pushed to S3.

For the conda environment and `.env` credential setup, see the top-level
[`README.md`](../../README.md). For how Energy Indicators compares to the other
three apps, see [`apps/RUN_GUIDE.md`](../RUN_GUIDE.md).

Prefer a plain shell script over a notebook? See [`run_example.sh`](run_example.sh)
for the same three steps with no narration, just commands and a `SOURCE`/`TARGET`
pair to edit at the top.

## Files in this app

| File | Purpose |
|---|---|
| `catalog_config.yaml` | ALL Energy Indicators input: collection metadata, paths, simulation keys, datacube extension |
| `staging/stage.py` | Builds the catalog skeleton: `collection.json` + `data/YYYY/MM/<item>/` folders for every simulation × month |
| `staging/move.py` | Links/copies/moves a whole month's source NetCDF files into the staged `YYYY/MM/item/` folder |
| `staging/update.py` | Generates STAC item metadata for the staged collection |
| `staging/delete_s3_collection_prefix.py` | Maintenance utility to delete a collection prefix from the S3 bucket |
| `staging/check_data.sh` | Optional pre-staging QC: counts expected daily/monthly files per year |
| `run_pipeline.sh` | Wraps `stage.py` + `update.py` + upload into one command (does **not** run `move.py`) |

Unlike HydroLand, **there is no data_type split**: every NC file produced for a given
month (5 daily indicators + the monthly aggregate stats) ends up as an asset of the
*same* monthly item — one item folder per month, not one per data type.

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml

APP_DIR = Path.cwd()
assert (APP_DIR / "catalog_config.yaml").exists(), "run this notebook from apps/energy_indicators/"

COLLECTION_ID = "EO.BSC.DAT.ENERGY_INDICATORS"
SANDBOX = Path(tempfile.mkdtemp(prefix="energy_notebook_"))
ENV = {**os.environ, "CATALOG_HOME_DIR": str(SANDBOX)}
print(f"Sandbox: {SANDBOX}")

def run(*args, check=True):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=APP_DIR, env=ENV, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")  # stage.py/move.py/update.py log to stderr
    if check:
        result.check_returncode()

Sandbox: /tmp/energy_notebook_9cdmabcp


## 1. The config file

`catalog_home_dir` in the committed yaml is the **real BSC HPC path** — it doesn't
exist on this machine, which is exactly why `CATALOG_HOME_DIR` is overridden to the
sandbox above instead. `overrides:` forces `model`/`experiment` to lowercase
(`ifs-nemo`/`ssp3-7.0`) to match the already-published folder naming convention on
S3, even though the shared `apps/_shared/simulations.yaml` defines them uppercase.

In [2]:
config = yaml.safe_load((APP_DIR / "catalog_config.yaml").read_text())
print("id:               ", config["id"])
print("license:          ", config["license"])
print("item_folder_level:", config["item_folder_level"])
print("property_order:   ", config["property_order"])
print("simulations:")
for sim in config["simulations"]:
    print(" -", sim)
print("overrides:        ", config["overrides"])
print("\napp_data_dir:     ", config["app_data_dir"], " (real BSC path — does not exist here)")
print("catalog_home_dir: ", config["catalog_home_dir"], " (overridden to the sandbox for this notebook)")

id:                EO.BSC.DAT.ENERGY_INDICATORS
license:           CC-BY-4.0
item_folder_level: MM
property_order:    ['model', 'experiment', 'generation', 'realization', 'activity', 'resolution']
simulations:
 - ifs-nemo_generation-2_projection
overrides:         {'model': 'ifs-nemo', 'experiment': 'ssp3-7.0', 'resolution': 'high'}

app_data_dir:      /gpfs/scratch/ehpc01/bsc032056/a314/output/energy_indicators/20150101/member01  (real BSC path — does not exist here)
catalog_home_dir:  /gpfs/scratch/ehpc01/bsc032056/a314/output/energy_indicators/20150101/member01  (overridden to the sandbox for this notebook)


## 2. Step 1 — `stage.py`: build the catalog skeleton

Pure config — no NC files are read. Creates `metadata/collection.json` +
`metadata/collection_config.json`, and one `data/YYYY/MM/<item>/` folder for every
simulation × month (no data_type split — only the projection run is enabled, so
this builds 420 monthly folders, 2015–2049).

In [3]:
run(sys.executable, "staging/stage.py")  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py


INFO | Collection: EO.BSC.DAT.ENERGY_INDICATORS (1 simulations)
INFO | [DRY RUN] Would write collection.json + collection_config.json
INFO | [DRY RUN] Skeleton ifs-nemo ssp3-7.0 (2015-2049): would create 420 folders
          Example: /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/data/2015/01/EO.BSC.DAT.ENERGY_INDICATORS_20150101T000000_20150131T230000__ifs-nemo__ssp3-7.0__2__1__projections__high
INFO | Dry run complete. Re-run with --execute to apply.


In [4]:
run(sys.executable, "staging/stage.py", "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py --execute


INFO | Collection: EO.BSC.DAT.ENERGY_INDICATORS (1 simulations)
INFO | Wrote /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/metadata/collection.json and /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/metadata/collection_config.json
INFO | Skeleton ifs-nemo ssp3-7.0 (2015-2049): 420 folders created


## 3. Step 2 — `move.py`: bring the NC files in

The raw simulation output is a flat tree, **one folder per month**, with no item
naming at all — `move.py` builds the item folder name from CLI flags and moves/links
every `.nc` file in that month's folder in, unchanged.

```
<raw_output_root>/
└── 2049/
    └── 06/
        ├── 2049_06_01_T00_00_cdd.nc        ← daily indicators (cdd, hdd, hwe, lwe, pvp)
        ├── ...
        └── checkpoint_cf_i_monthly_timestep_60_monthly_histogram.000005-000008.pkl  ← ignored (not .nc)
```

In [ ]:
ENERGY_SOURCE = "/work/lealroja/applications-stac-catalog-phase3-energy-output-example"  # <-- HARDCODED PATH: change to your own <root>/YYYY/MM source folder to re-run with different data

move_args = [
    sys.executable, "staging/move.py",
    "--source-dir", ENERGY_SOURCE,
    "--target-collection-dir", str(SANDBOX / COLLECTION_ID),
    "--model", "ifs-nemo", "--experiment", "ssp3-7.0", "--generation", "2",
    "--realization", "1", "--activity", "projections", "--resolution", "high",
    "--link",
]
run(*move_args)  # dry run

In [6]:
run(*move_args, "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-dir /work/lealroja/applications-stac-catalog-phase3-energy-output-example --target-collection-dir /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS --model ifs-nemo --experiment ssp3-7.0 --generation 2 --realization 1 --activity projections --resolution high --link --execute


INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-energy-output-example/2049/06/2049_06_01_T00_00_cdd.nc -> /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/data/2049/06/EO.BSC.DAT.ENERGY_INDICATORS_20490601T000000_20490630T230000__ifs-nemo__ssp3-7.0__2__1__projections__high/2049_06_01_T00_00_cdd.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-energy-output-example/2049/06/2049_06_01_T00_00_hdd.nc -> /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/data/2049/06/EO.BSC.DAT.ENERGY_INDICATORS_20490601T000000_20490630T230000__ifs-nemo__ssp3-7.0__2__1__projections__high/2049_06_01_T00_00_hdd.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-energy-output-example/2049/06/2049_06_01_T00_00_hwe.nc -> /tmp/energy_notebook_9cdmabcp/EO.BSC.DAT.ENERGY_INDICATORS/data/2049/06/EO.BSC.DAT.ENERGY_INDICATORS_20490601T000000_20490630T230000__ifs-nemo__ssp3-7.0__2__1__projections__high/2049_06_01_T00_00_hwe.nc
INFO

**Recognized:** any `*.nc` file directly inside a `<raw_output_root>/YYYY/MM/`
folder. **Ignored (not copied):** anything that isn't `.nc` — in particular the
`.pkl` checkpoint files left behind by incomplete/in-progress monthly aggregation
runs. `--source-dir` only walks exactly two levels (`YYYY/MM/`), not recursive.

## 4. Step 3 — `update.py`: generate STAC item metadata

In [7]:
run(sys.executable, "staging/update.py")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_id:EO.BSC.DAT.ENERGY_INDICATORS_20491201T000000_20491231T230000__ifs-nemo__ssp3-7.0__2__1__projections__high, collection_id:EO.BSC.DAT.ENERGY_INDICATORS, year:2049, month:12, day:01
item_id:EO.BSC.DAT.ENERGY_INDICATORS_20491101T000000_20491130T230000__ifs-nemo__ssp3-7.0__2__1__projections__high, collection_id:EO.BSC.DAT.ENERGY_INDICATORS, year:2049, month:11, day:01
item_id:EO.BSC.DAT.ENERGY_INDICATORS_20491001T000000_20491031T230000__ifs-nemo__ssp3-7.0__2__1__projections__high, collection_id:EO.BSC.DAT.ENERGY_INDICATORS, year:2049, month:10, day:01
item_id:EO.BSC.DAT.ENERGY_INDICATORS_20490901T000000_20490930T230000__ifs-nemo__ssp3-7.0__2__1__projections__high, collection_id:EO.BSC.DAT.ENERGY_INDICATORS, year:2049, month:09, day:01
item_id:EO.BSC.DAT.ENERGY_INDICATORS_20490801T000000_20490831T230000__ifs-nemo__ssp3-7.0__2__1__projections__high, collection_id:EO.BSC.DAT.ENERGY_INDICATORS, year:2049, month:08, day:01
item_id

## 5. Inspect the results

In [8]:
items_dir = SANDBOX / COLLECTION_ID / "metadata" / "items"
item_paths = sorted(items_dir.glob("*.json"))
print(f"{len(item_paths)} item JSON(s) generated:")
for p in item_paths:
    print(" ", p.name)

1 item JSON(s) generated:
  EO.BSC.DAT.ENERGY_INDICATORS_20490601T000000_20490630T230000__ifs-nemo__ssp3-7.0__2__1__projections__high.json


In [9]:
sample = json.loads(item_paths[0].read_text())
print("id:        ", sample["id"])
print("properties:", sample["properties"])
print(f"assets ({len(sample['assets'])}):", list(sample["assets"])[:5], "...")

id:         EO.BSC.DAT.ENERGY_INDICATORS_20490601T000000_20490630T230000__ifs-nemo__ssp3-7.0__2__1__projections__high
properties: {'model': 'ifs-nemo', 'experiment': 'ssp3-7.0', 'generation': '2', 'realization': '1', 'activity': 'projections', 'resolution': 'high', 'start_datetime': '2049-06-01T00:00:00Z', 'end_datetime': '2049-06-30T23:00:00Z', 'datetime': '2049-06-01T00:00:00Z'}
assets (40): ['2049_06_01_T00_00_cdd.nc', '2049_06_01_T00_00_hdd.nc', '2049_06_01_T00_00_hwe.nc', '2049_06_01_T00_00_lwe.nc', '2049_06_01_T00_00_pvp.nc'] ...


## 6. Upload (not run in this notebook)

Skipped here on purpose. To actually publish:

1. Copy `apps/energy_indicators/.env.example` → `apps/energy_indicators/.env` and
   fill in `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (see the top-level README's
   "Configure credentials" section).
2. Remove the `CATALOG_HOME_DIR` sandbox override (or set it to BSC's real path).
3. Run:
   ```bash
   bash run_pipeline.sh --execute --upload
   bash run_pipeline.sh --execute --upload --year 2016   # resume/retry a single year
   ```

Note: `catalog_config.yaml` is marked provisional in places — only the projection
run is enabled (`simulations:`), the historical run is commented out pending BSC
confirmation.

In [10]:
import shutil
shutil.rmtree(SANDBOX)
print(f"Removed sandbox: {SANDBOX}")

Removed sandbox: /tmp/energy_notebook_9cdmabcp
